# Python: jour 2

In [1]:
import pandas as pd
import geopandas as gpd
import folium
import re
from datetime import datetime, date, time

## Encodages:
- ASCII
- latin 1: ISO-8859-1
- ISO-8859-15
- Windows 1252 (alias CP1252, ANSI)
- Unicode : UTF-8, UTF-16, UTF-32, .. 

In [2]:
texte = "Le cœur  de la ville L'Haÿ-les-Roses n'a pas de prix en €"
print(texte)

Le cœur  de la ville L'Haÿ-les-Roses n'a pas de prix en €


In [3]:
texte.encode('UTF-8')

b"Le c\xc5\x93ur  de la ville L'Ha\xc3\xbf-les-Roses n'a pas de prix en \xe2\x82\xac"

In [4]:
texte.encode('cp1252')

b"Le c\x9cur  de la ville L'Ha\xff-les-Roses n'a pas de prix en \x80"

In [5]:
texte.encode('ISO-8859-15')

b"Le c\xbdur  de la ville L'Ha\xff-les-Roses n'a pas de prix en \xa4"

In [6]:
# UnicodeEncodeError: 'latin-1' codec can't encode character '\u0153' in position 4: ordinal not in range(256)
# texte.encode('ISO-8859-1')

In [7]:
code = b"\xe2\x82\xac"
print(code.decode('UTF-8'))
print(code.decode('cp1252'))

€
â‚¬


## Communes de France

In [8]:
dtype = {
    'code_insee': str,
    'code_postal': str,
    'reg_code': str,
    'dep_code': str
}
df_villes = pd.read_csv('data/villes_fr.csv', encoding='cp1252', sep=';', dtype=dtype, decimal=',')
df_villes.head()

,code_insee,code_postal,nom_standard,nom_sans_pronom,dep_code,dep_nom,reg_code,reg_nom,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
0,01001,01400,L'Abergement-Clémenciat,Abergement-Clémenciat,01,Ain,84,Auvergne-Rhône-Alpes,832,1565,53.0,242,206,272,46.151,4.921,46.153,4.926
1,01002,01640,L'Abergement-de-Varey,Abergement-de-Varey,01,Ain,84,Auvergne-Rhône-Alpes,267,912,29.0,483,290,748,46.007,5.423,46.009,5.428
2,01004,01500,Ambérieu-en-Bugey,Ambérieu-en-Bugey,01,Ain,84,Auvergne-Rhône-Alpes,14854,2448,607.0,379,237,753,45.958,5.360,45.961,5.373
3,01005,01330,Ambérieux-en-Dombes,Ambérieux-en-Dombes,01,Ain,84,Auvergne-Rhône-Alpes,1897,1605,118.0,290,265,302,45.996,4.903,45.996,4.912
4,01006,01300,Ambléon,Ambléon,01,Ain,84,Auvergne-Rhône-Alpes,113,602,19.0,589,330,940,45.748,5.601,45.750,5.594


In [9]:
type(df_villes)

pandas.DataFrame

In [10]:
df_villes.info()

<class 'pandas.DataFrame'>
RangeIndex: 34935 entries, 0 to 34934
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code_insee          34935 non-null  str    
 1   code_postal         34932 non-null  str    
 2   nom_standard        34935 non-null  str    
 3   nom_sans_pronom     34935 non-null  str    
 4   dep_code            34935 non-null  str    
 5   dep_nom             34935 non-null  str    
 6   reg_code            34935 non-null  str    
 7   reg_nom             34935 non-null  str    
 8   population          34935 non-null  int64  
 9   superficie_hectare  34935 non-null  int64  
 10  densite             34932 non-null  float64
 11  altitude_moyenne    34935 non-null  int64  
 12  altitude_minimale   34935 non-null  int64  
 13  altitude_maximale   34935 non-null  int64  
 14  latitude_mairie     34935 non-null  float64
 15  longitude_mairie    34935 non-null  float64
 16  latitude_centre

In [11]:
df_villes.describe()

,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
count,3.493500e+04,3.493500e+04,34932.000000,34935.000000,34935.000000,34935.000000,34935.000000,34935.000000,34926.000000,34926.000000
mean,1.936405e+03,1.763584e+03,169.279056,279.172606,188.710405,384.897782,46.787217,2.714014,46.786757,2.714199
std,1.498968e+04,1.481510e+04,752.070774,288.589425,201.561722,452.958268,3.586378,4.410897,3.588029,4.411771
min,0.000000e+00,2.000000e+00,0.000000,0.000000,0.000000,0.000000,-21.379000,-61.794000,-21.340000,-61.780000
25%,1.970000e+02,6.490000e+02,18.000000,105.000000,55.000000,132.000000,45.070000,0.793500,45.069000,0.793000
50%,4.570000e+02,1.088000e+03,41.000000,189.000000,134.000000,231.000000,47.351000,2.718000,47.353000,2.718000
75%,1.164000e+03,1.867000e+03,100.000000,337.000000,252.000000,436.500000,48.824000,4.915500,48.823000,4.918000
max,2.133111e+06,1.871833e+06,28220.000000,2713.000000,9589.000000,4808.000000,51.071000,55.793000,51.073000,55.754000


In [12]:
# Series : column (or row)
df_villes['nom_standard']

0        L'Abergement-Clémenciat
1          L'Abergement-de-Varey
2              Ambérieu-en-Bugey
3            Ambérieux-en-Dombes
4                        Ambléon
                  ...           
34930              M'Tsangamouji
34931                   Ouangani
34932                   Pamandzi
34933                       Sada
34934                   Tsingoni
Name: nom_standard, Length: 34935, dtype: str

In [13]:
df_villes.nom_standard

0        L'Abergement-Clémenciat
1          L'Abergement-de-Varey
2              Ambérieu-en-Bugey
3            Ambérieux-en-Dombes
4                        Ambléon
                  ...           
34930              M'Tsangamouji
34931                   Ouangani
34932                   Pamandzi
34933                       Sada
34934                   Tsingoni
Name: nom_standard, Length: 34935, dtype: str

In [14]:
# liste de colonnes
df_villes[['nom_standard', 'code_postal']]

,nom_standard,code_postal
0,L'Abergement-Clémenciat,01400
1,L'Abergement-de-Varey,01640
2,Ambérieu-en-Bugey,01500
3,Ambérieux-en-Dombes,01330
4,Ambléon,01300
...,...,...
34930,M'Tsangamouji,97650
34931,Ouangani,97670
34932,Pamandzi,97615
34933,Sada,97640


In [15]:
df_villes.iloc[:3, :4]

,code_insee,code_postal,nom_standard,nom_sans_pronom
0,01001,01400,L'Abergement-Clémenciat,Abergement-Clémenciat
1,01002,01640,L'Abergement-de-Varey,Abergement-de-Varey
2,01004,01500,Ambérieu-en-Bugey,Ambérieu-en-Bugey


In [16]:
# derniere ligne : Series
df_villes.iloc[-1]

code_insee               97617
code_postal              97680
nom_standard          Tsingoni
nom_sans_pronom       Tsingoni
dep_code                   976
dep_nom                Mayotte
reg_code                     6
reg_nom                Mayotte
population               13934
superficie_hectare        3426
densite                  407.0
altitude_moyenne           147
altitude_minimale            0
altitude_maximale          480
latitude_mairie         -12.79
longitude_mairie        45.105
latitude_centre        -12.782
longitude_centre        45.134
Name: 34934, dtype: object

In [17]:
df_villes.iloc[20_000:20_005]

,code_insee,code_postal,nom_standard,nom_sans_pronom,dep_code,dep_nom,reg_code,reg_nom,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
20000,55001,55130,Abainville,Abainville,55,Meuse,44,Grand Est,285,1360,21.0,323,282,388,48.530,5.494,48.533,5.515
20001,55002,55400,Abaucourt-Hautecourt,Abaucourt-Hautecourt,55,Meuse,44,Grand Est,99,970,10.0,224,210,251,49.197,5.540,49.197,5.549
20002,55004,55110,Aincreville,Aincreville,55,Meuse,44,Grand Est,81,910,9.0,235,187,314,49.368,5.121,49.370,5.116
20003,55005,55130,Amanty,Amanty,55,Meuse,44,Grand Est,35,1118,3.0,389,295,426,48.518,5.613,48.515,5.604
20004,55007,55300,Ambly-sur-Meuse,Ambly-sur-Meuse,55,Meuse,44,Grand Est,237,638,37.0,232,204,273,49.019,5.443,49.023,5.452


In [18]:
df_villes.iloc[-1:]

,code_insee,code_postal,nom_standard,nom_sans_pronom,dep_code,dep_nom,reg_code,reg_nom,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
34934,97617,97680,Tsingoni,Tsingoni,976,Mayotte,6,Mayotte,13934,3426,407.0,147,0,480,-12.79,45.105,-12.782,45.134


Opérateurs de comparaisons:
- `==`, `!=`
- `<`, `<=`, `>`, `>=`

In [19]:
df_villes.loc[df_villes.population >= 500_000]

,code_insee,code_postal,nom_standard,nom_sans_pronom,dep_code,dep_nom,reg_code,reg_nom,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
4342,13055,13000,Marseille,Marseille,13,Bouches-du-Rhône,93,Provence-Alpes-Côte d'Azur,873076,24060,3629.0,20,0,652,43.296,5.370,NaN,NaN
11793,31555,31100,Toulouse,Toulouse,31,Haute-Garonne,76,Occitanie,504078,11809,4269.0,148,115,263,43.605,1.444,43.596,1.432
27082,69123,69000,Lyon,Lyon,69,Rhône,84,Auvergne-Rhône-Alpes,522250,4790,10903.0,237,162,305,45.764,4.836,NaN,NaN
29244,75056,75000,Paris,Paris,75,Paris,11,Île-de-France,2133111,10540,20238.0,35,28,131,48.857,2.352,NaN,NaN


In [20]:
df_villes.population >= 500_000

0        False
1        False
2        False
3        False
4        False
         ...  
34930    False
34931    False
34932    False
34933    False
34934    False
Name: population, Length: 34935, dtype: bool

In [21]:
df_villes.population.sum()

np.int64(67648309)

In [22]:
df_villes.loc[
    # rows : 'WHERE'
    df_villes.population >= 500_000,
    # columns
    ['nom_standard', 'dep_code', 'population']
].sort_values('population', ascending=False)

,nom_standard,dep_code,population
29244,Paris,75,2133111
4342,Marseille,13,873076
27082,Lyon,69,522250
11793,Toulouse,31,504078


In [23]:
# - villes à 0 habitants
df_villes.loc[
    df_villes.population == 0,
    ['dep_code', 'nom_standard', 'population']
]

,dep_code,nom_standard,population
20032,55,Beaumont-en-Verdunois,0
20043,55,Bezonvaux,0
20118,55,Cumières-le-Mort-Homme,0
20161,55,Fleury-devant-Douaumont,0
20198,55,Haumont-près-Samogneux,0
20257,55,Louvemont-Côte-du-Poivre,0
23101,60,Les Hauts-Talican,0
32700,85,L'Oie,0
32739,85,Sainte-Florence,0


In [24]:
# - population totale du 31
# draft
df_villes.loc[
    df_villes.dep_code == '31',
    ['dep_code', 'nom_standard', 'population']
] #.sum()

,dep_code,nom_standard,population
11246,31,Agassac,119
11247,31,Aignes,239
11248,31,Aigrefeuille,1280
11249,31,Ayguesvives,2764
11250,31,Alan,287
...,...,...,...
11827,31,Villenouvelle,1454
11828,31,Binos,44
11829,31,Escoulis,78
11830,31,Larra,2181


In [25]:
df_villes.loc[
    df_villes.dep_code == '31',
    'population'
].sum()

np.int64(1434367)

In [26]:
# - nom standard de la ville avec le plus d'habitants
max_population = df_villes.population.max()
ville = df_villes.loc[
    df_villes.population == max_population,
    'nom_standard'
].iloc[0]
print(f"{ville} a le maximum de population {max_population}")

Paris a le maximum de population 2133111


In [27]:
dep = '31'
df_villes31 = df_villes.loc[df_villes.dep_code == '31']
max_population = df_villes31.population.max()
ville = df_villes31.loc[
    df_villes31.population == max_population,
    'nom_standard'
].iloc[0]
print(f"{ville} a le maximum de population {max_population} dans le {dep}")


Toulouse a le maximum de population 504078 dans le 31


Combinaison logique sur des booléens:
- and
- or
- not


In [28]:
city = "L'Haÿ-les-Roses"
len(city) >= 5 and 'ÿ' in city

True

Combinaison logique en mode vectoriel:
- `&` : ET element à element (element-wise)
- `|` : OR element à element (element-wise)
- `~` : NOT element à element (element-wise)

In [29]:
# - villes du 31 de plus de ?k habitants
population_threshold = 20_000
selection_dep31_population = df_villes.loc[
    (df_villes.dep_code == '31')
    & (df_villes.population >= population_threshold)
]
selection_dep31_population

,code_insee,code_postal,nom_standard,nom_sans_pronom,dep_code,dep_nom,reg_code,reg_nom,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
11313,31069,31700,Blagnac,Blagnac,31,Haute-Garonne,76,Occitanie,26466,1720,1539.0,142,119,153,43.635,1.398,43.642,1.379
11393,31149,31770,Colomiers,Colomiers,31,Haute-Garonne,76,Occitanie,40159,2104,1909.0,172,145,191,43.611,1.335,43.612,1.327
11400,31157,31270,Cugnaux,Cugnaux,31,Haute-Garonne,76,Occitanie,20341,1302,1562.0,164,150,170,43.537,1.345,43.545,1.343
11634,31395,31600,Muret,Muret,31,Haute-Garonne,76,Occitanie,25060,5837,429.0,189,152,305,43.462,1.331,43.449,1.308
11793,31555,31100,Toulouse,Toulouse,31,Haute-Garonne,76,Occitanie,504078,11809,4269.0,148,115,263,43.605,1.444,43.596,1.432
11795,31557,31170,Tournefeuille,Tournefeuille,31,Haute-Garonne,76,Occitanie,29439,1826,1612.0,164,146,190,43.583,1.347,43.578,1.335


In [30]:
# - Villes qui commencent par Tou
# - Villes qui contiennent ÿ
# - Villes qui n'ont pas de coordonnées du centre

In [31]:
df_villes.nom_standard.str.upper()

0        L'ABERGEMENT-CLÉMENCIAT
1          L'ABERGEMENT-DE-VAREY
2              AMBÉRIEU-EN-BUGEY
3            AMBÉRIEUX-EN-DOMBES
4                        AMBLÉON
                  ...           
34930              M'TSANGAMOUJI
34931                   OUANGANI
34932                   PAMANDZI
34933                       SADA
34934                   TSINGONI
Name: nom_standard, Length: 34935, dtype: str

In [32]:
df_villes.loc[
    df_villes.nom_standard.str.startswith('Toul')
].sort_values('nom_standard')

,code_insee,code_postal,nom_standard,nom_sans_pronom,dep_code,dep_nom,reg_code,reg_nom,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
19925,54528,54200,Toul,Toul,54,Meurthe-et-Moselle,44,Grand Est,15849,3095,512.0,229,200,400,48.676,5.894,48.686,5.895
2338,07323,07130,Toulaud,Toulaud,07,Ardèche,84,Auvergne-Rhône-Alpes,1714,3528,49.0,347,150,631,44.897,4.820,44.902,4.793
12805,33533,33210,Toulenne,Toulenne,33,Gironde,75,Nouvelle-Aquitaine,2813,655,429.0,17,1,34,44.559,-0.262,44.564,-0.275
2767,08454,08430,Touligny,Touligny,08,Ardennes,44,Grand Est,83,726,11.0,242,0,0,49.671,4.617,49.677,4.615
1103,02745,02250,Toulis-et-Attencourt,Toulis-et-Attencourt,02,Aisne,32,Hauts-de-France,113,783,14.0,79,66,109,49.700,3.744,49.697,3.750
32391,83137,83000,Toulon,Toulon,83,Var,93,Provence-Alpes-Côte d'Azur,180452,4419,4084.0,126,0,584,43.120,5.932,43.136,5.932
1472,03286,03400,Toulon-sur-Allier,Toulon-sur-Allier,03,Allier,84,Auvergne-Rhône-Alpes,1157,3925,29.0,241,206,284,46.518,3.360,46.511,3.377
28292,71542,71320,Toulon-sur-Arroux,Toulon-sur-Arroux,71,Saône-et-Loire,27,Bourgogne-Franche-Comté,1458,4368,33.0,297,245,391,46.693,4.140,46.683,4.148
4265,12281,12200,Toulonjac,Toulonjac,12,Aveyron,76,Occitanie,749,747,100.0,345,288,441,44.380,2.000,44.380,1.999
26068,66213,66350,Toulouges,Toulouges,66,Pyrénées-Orientales,76,Occitanie,7334,802,914.0,66,50,82,42.665,2.838,42.670,2.824


In [33]:
df_villes.loc[
    df_villes.dep_code.isin(['31', '64'])
]

,code_insee,code_postal,nom_standard,nom_sans_pronom,dep_code,dep_nom,reg_code,reg_nom,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
11246,31001,31230,Agassac,Agassac,31,Haute-Garonne,76,Occitanie,119,961,12.0,262,197,336,43.371,0.887,43.372,0.885
11247,31002,31550,Aignes,Aignes,31,Haute-Garonne,76,Occitanie,239,2205,11.0,253,198,323,43.320,1.588,43.335,1.586
11248,31003,31280,Aigrefeuille,Aigrefeuille,31,Haute-Garonne,76,Occitanie,1280,464,276.0,187,151,230,43.568,1.589,43.569,1.584
11249,31004,31450,Ayguesvives,Ayguesvives,31,Haute-Garonne,76,Occitanie,2764,1336,207.0,201,156,271,43.438,1.597,43.429,1.594
11250,31005,31420,Alan,Alan,31,Haute-Garonne,76,Occitanie,287,1146,25.0,377,271,520,43.230,0.940,43.220,0.928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25389,64556,64150,Vielleségure,Vielleségure,64,Pyrénées-Atlantiques,75,Nouvelle-Aquitaine,387,1428,27.0,201,115,275,43.357,-0.683,43.356,-0.702
25390,64557,64410,Vignes,Vignes,64,Pyrénées-Atlantiques,75,Nouvelle-Aquitaine,449,801,56.0,178,102,243,43.526,-0.412,43.523,-0.410
25391,64558,64990,Villefranque,Villefranque,64,Pyrénées-Atlantiques,75,Nouvelle-Aquitaine,2893,1722,168.0,49,0,131,43.437,-1.453,43.442,-1.448
25392,64559,64130,Viodos-Abense-de-Bas,Viodos-Abense-de-Bas,64,Pyrénées-Atlantiques,75,Nouvelle-Aquitaine,748,1281,58.0,195,110,420,43.242,-0.882,43.248,-0.897


In [34]:
df_villes.loc[
    df_villes.nom_standard.str.contains('ÿ', case=False),
    ['dep_code', 'nom_standard']
]

,dep_code,nom_standard
900,02,Moÿ-de-l'Aisne
18157,51,Aÿ-Champagne
30120,77,Faÿ-lès-Nemours
34592,94,L'Haÿ-les-Roses


In [35]:
# villes avec au moins 2 tirets
df_villes.loc[
    df_villes.nom_standard.str.contains(r'-.*-', case=False),
    ['dep_code', 'nom_standard']
]

,dep_code,nom_standard
1,01,L'Abergement-de-Varey
2,01,Ambérieu-en-Bugey
3,01,Ambérieux-en-Dombes
7,01,Andert-et-Condon
17,01,Ars-sur-Formans
...,...,...
34834,971,Terre-de-Haut
34845,972,Fonds-Saint-Denis
34846,972,Fort-de-France
34882,973,Saint-Laurent-du-Maroni


In [36]:
# villes avec au moins 5 tirets
selection = df_villes.loc[
    df_villes.nom_standard.str.count('-') >= 5,
    ['dep_code', 'nom_standard', 'population']
]
selection['nb_tirets'] = selection.nom_standard.str.count('-')
selection.sort_values(['nb_tirets', 'population'], ascending=[True, False])

,dep_code,nom_standard,population,nb_tirets
27679,70,Scey-sur-Saône-et-Saint-Albin,1530,5
29279,76,Les Authieux-sur-le-Port-Saint-Ouen,1230,5
15844,42,Saint-Jean-Saint-Maurice-sur-Loire,1155,5
27290,70,Beaujeu-Saint-Vallier-Pierrejux-et-Quitteur,923,5
11155,30,Saint-Jean-de-Maruéjols-et-Avéjan,852,5
8052,24,Javerlhac-et-la-Chapelle-Saint-Robert,823,5
10998,30,Durfort-et-Saint-Martin-de-Sossenac,753,5
4834,14,Saint-Martin-de-Bienfaite-la-Cressonnière,499,5
23395,61,Saint-Evroult-Notre-Dame-du-Bois,440,5
32768,85,Saint-Martin-Lars-en-Sainte-Hermine,419,5


In [37]:
max_letters = df_villes.nom_standard.str.len().max()
df_villes.loc[df_villes.nom_standard.str.len() == max_letters]

,code_insee,code_postal,nom_standard,nom_sans_pronom,dep_code,dep_nom,reg_code,reg_nom,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
18604,51513,51290,Saint-Remy-en-Bouzemont-Saint-Genest-et-Isson,Saint-Remy-en-Bouzemont-Saint-Genest-et-Isson,51,Marne,44,Grand Est,491,2204,22.0,116,109,131,48.628,4.644,48.613,4.649


In [38]:
df_villes.columns

Index(['code_insee', 'code_postal', 'nom_standard', 'nom_sans_pronom',
       'dep_code', 'dep_nom', 'reg_code', 'reg_nom', 'population',
       'superficie_hectare', 'densite', 'altitude_moyenne',
       'altitude_minimale', 'altitude_maximale', 'latitude_mairie',
       'longitude_mairie', 'latitude_centre', 'longitude_centre'],
      dtype='str')

In [39]:
columns = [
    'dep_code', 'nom_standard', 'population',
    'latitude_mairie', 'longitude_mairie', 
    'latitude_centre', 'longitude_centre'
]
df_villes.loc[
    df_villes.longitude_centre.isna()
    | df_villes.latitude_centre.isna(),
    columns
].sort_values('population', ascending=False)

,dep_code,nom_standard,population,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre
29244,75,Paris,2133111,48.857,2.352,NaN,NaN
4342,13,Marseille,873076,43.296,5.370,NaN,NaN
27082,69,Lyon,522250,45.764,4.836,NaN,NaN
29328,76,Bihorel,8199,49.454,1.116,NaN,NaN
29810,76,Saint-Lucien,233,49.509,1.448,NaN,NaN
20117,55,Culey,125,48.755,5.267,NaN,NaN
23101,60,Les Hauts-Talican,0,49.329,2.004,NaN,NaN
32700,85,L'Oie,0,46.798,-1.130,NaN,NaN
32739,85,Sainte-Florence,0,46.797,-1.152,NaN,NaN


## Une jolie carte pour finir

In [40]:
gdf_big_villes_31 = gpd.GeoDataFrame(
    selection_dep31_population,
    geometry=gpd.points_from_xy(selection_dep31_population.longitude_mairie, selection_dep31_population.latitude_mairie),
    crs='EPSG:4326'
)
gdf_big_villes_31

,code_insee,code_postal,nom_standard,nom_sans_pronom,dep_code,dep_nom,reg_code,reg_nom,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre,geometry
11313,31069,31700,Blagnac,Blagnac,31,Haute-Garonne,76,Occitanie,26466,1720,1539.0,142,119,153,43.635,1.398,43.642,1.379,POINT (1.398 43.635)
11393,31149,31770,Colomiers,Colomiers,31,Haute-Garonne,76,Occitanie,40159,2104,1909.0,172,145,191,43.611,1.335,43.612,1.327,POINT (1.335 43.611)
11400,31157,31270,Cugnaux,Cugnaux,31,Haute-Garonne,76,Occitanie,20341,1302,1562.0,164,150,170,43.537,1.345,43.545,1.343,POINT (1.345 43.537)
11634,31395,31600,Muret,Muret,31,Haute-Garonne,76,Occitanie,25060,5837,429.0,189,152,305,43.462,1.331,43.449,1.308,POINT (1.331 43.462)
11793,31555,31100,Toulouse,Toulouse,31,Haute-Garonne,76,Occitanie,504078,11809,4269.0,148,115,263,43.605,1.444,43.596,1.432,POINT (1.444 43.605)
11795,31557,31170,Tournefeuille,Tournefeuille,31,Haute-Garonne,76,Occitanie,29439,1826,1612.0,164,146,190,43.583,1.347,43.578,1.335,POINT (1.347 43.583)


In [41]:

minx, miny, maxx, maxy = gdf_big_villes_31.total_bounds
centre = [(miny + maxy) / 2, (minx + maxx) / 2]

m = folium.Map(location=centre, zoom_start=9, tiles='CartoDB positron')

folium.GeoJson(
    gdf_big_villes_31,
    tooltip=folium.GeoJsonTooltip(
        fields=['nom_standard', 'population', 'superficie_hectare'],
        aliases=['Ville', 'Population', 'Superficie (Ha)']
    )
).add_to(m)
m

### Calculs de distance

In [57]:
df_villes_31 =  df_villes.loc[df_villes.dep_code == '31']
gdf_villes_31 = gpd.GeoDataFrame(
    df_villes_31,
    geometry=gpd.points_from_xy(df_villes_31.longitude_centre, df_villes_31.latitude_centre),
    crs='EPSG:4326'
).to_crs('EPSG:2154')  # Lambert 93: metropole

In [58]:
toulouse_pt = gdf_villes_31.loc[gdf_villes_31.nom_standard == 'Toulouse', 'geometry'].iloc[0]
print(toulouse_pt)

POINT (573353.5109032689 6278695.731091948)


In [59]:
distance_threshold = 10_000 # meters
gdf_villes_31['distance_tlse'] = gdf_villes_31.geometry.distance(toulouse_pt)
gdf_villes_proches = gdf_villes_31.loc[
    gdf_villes_31.distance_tlse.between(0.1, distance_threshold),
    ['code_postal', 'nom_standard', 'population', 'distance_tlse']
].sort_values('distance_tlse')
gdf_villes_proches

,code_postal,nom_standard,population,distance_tlse
11288,31130,Balma,17385,6158.069808
11313,31700,Blagnac,26466,6667.030311
11684,31520,Ramonville-Saint-Agne,14949,6871.689373
11813,31320,Vieille-Toulouse,1217,7573.280366
11799,31240,L'Union,12358,7692.582899
11672,31120,Portet-sur-Garonne,9805,7831.392241
11795,31170,Tournefeuille,29439,8087.707423
11650,31320,Pechbusque,989,8145.769670
11266,31140,Aucamville,9349,8360.308532
11523,31140,Launaguet,9260,8431.158430
